# SEC-BERT calibration export v1
Attach the private Kaggle Dataset containing `sec_bert_calibration_kaggle_bundle.zip`, enable a GPU and Internet, then run all cells. This retrains the frozen recipe and exports untouched 608-sample calibration logits plus the selected checkpoint.

In [ ]:
from pathlib import Path
import shutil, zipfile
input_root = Path('/kaggle/input')
project = Path('/kaggle/working/selective-audit-routing')
if project.exists(): shutil.rmtree(project)
scripts = sorted(input_root.rglob('train_sec_bert_calibration_export_v1.py'))
if len(scripts) == 1:
    source_root = scripts[0].parent.parent
    shutil.copytree(source_root, project)
    print('Copied Kaggle-unpacked dataset:', source_root)
else:
    all_zips = sorted(input_root.rglob('*.zip'))
    bundles = [path for path in all_zips if 'sec_bert_calibration' in path.name.lower()]
    if not bundles and len(all_zips) == 1: bundles = all_zips
    assert len(bundles) == 1, (f'Expected one unpacked calibration script or bundle; scripts={scripts}, zips={all_zips}')
    project.mkdir(parents=True)
    with zipfile.ZipFile(bundles[0]) as archive: archive.extractall(project)
    print('Extracted ZIP bundle:', bundles[0])
assert (project/'configs/sec_bert_calibration_export_v1.json').exists()
assert (project/'data/processed/model_dataset_v2.csv').exists()
print('Project ready:', project)

In [ ]:
import subprocess, sys
command = [sys.executable, str(project/'src/train_sec_bert_calibration_export_v1.py'), '--cache-dir', '/kaggle/working/hf_cache']
subprocess.run(command, cwd=project, check=True)

In [ ]:
import json, math, sys
import numpy as np
import pandas as pd
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForSequenceClassification, AutoTokenizer
sys.path.insert(0, str(project/'src'))
from train_sec_bert_calibration_export_v1 import TextDataset, make_collator, predict, clean_mask
checkpoint = project/'artifacts/models/sec_bert_calibration_export_v1'
assert (checkpoint/'model.safetensors').exists(), 'Checkpoint missing: rerun the training cell first.'
config = json.loads((project/'configs/sec_bert_calibration_export_v1.json').read_text())
frame = pd.read_csv(project/config['dataset'])
test = frame.loc[clean_mask(frame, 'test')].copy().reset_index(drop=True)
assert len(test) == 890, len(test)
labels = sorted(frame[config['target']].dropna().astype(str).unique())
label_to_id = {label: index for index, label in enumerate(labels)}
id_to_label = {index: label for label, index in label_to_id.items()}
test[config['text_column']] = test[config['text_column']].fillna('').astype(str)
test['label_id'] = test[config['target']].map(label_to_id).astype(int)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
tokenizer = AutoTokenizer.from_pretrained(checkpoint)
model = AutoModelForSequenceClassification.from_pretrained(checkpoint).to(device)
loader = DataLoader(TextDataset(test[config['text_column']].tolist(), test['label_id'].tolist()), batch_size=8, shuffle=False, collate_fn=make_collator(tokenizer, int(config['max_length'])), num_workers=0, pin_memory=device.type == 'cuda')
logits, y = predict(model, loader, device)
prob = torch.softmax(torch.tensor(logits), dim=1).numpy()
pred_id = prob.argmax(axis=1)
pred = np.asarray([id_to_label[int(index)] for index in pred_id])
ordered = np.sort(prob, axis=1)
clipped = np.clip(prob, 1e-15, 1.0)
out = test[['sample_id','report_id',config['target'],'context_quality']].copy()
out['prediction'] = pred
out['confidence'] = prob.max(axis=1)
out['margin'] = ordered[:,-1] - ordered[:,-2]
out['entropy'] = -(clipped*np.log(clipped)).sum(axis=1)
out['correct'] = pred_id == y
out['logits_json'] = [json.dumps(list(map(float,row)), separators=(',',':')) for row in logits]
out['probability_json'] = [json.dumps(dict(zip(labels,map(float,row))), ensure_ascii=False, sort_keys=True, separators=(',',':')) for row in prob]
table_dir = project/'results/tables'
table_dir.mkdir(parents=True, exist_ok=True)
out.to_csv(table_dir/'sec_bert_retrospective_test_logits_v1.csv', index=False)
print({'samples': len(out), 'accuracy': float(out.correct.mean()), 'device': str(device)})

In [ ]:
result_zip = shutil.make_archive('/kaggle/working/sec_bert_calibration_results', 'zip', project, 'results')
checkpoint_zip = shutil.make_archive('/kaggle/working/sec_bert_calibration_checkpoint', 'zip', project, 'artifacts/models/sec_bert_calibration_export_v1')
print(result_zip)
print(checkpoint_zip)